# E3: graph degree features and leakage (RQ3)

Approaches: **A** labelled-only edges (dissertation), **B** full network (Kaggle), **C** notebook cell 39 construction, **C2** corrected time-bounded control.

Model: Random Forest with the notebook's settings, 5 seeds. Runs on CPU in about 8 minutes.

## 0. Upload the Kaggle files for approach B
Upload `elliptic_txs_edgelist.csv` (and optionally `elliptic_txs_classes.csv`). Skip this cell to run A, C and C2 only.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose elliptic_txs_edgelist.csv
print(list(uploaded))

Saving elliptic_txs_edgelist.cvs.csv to elliptic_txs_edgelist.cvs (1).csv
['elliptic_txs_edgelist.cvs (1).csv']


## 1. Setup and data

In [ ]:
# %% 1. Setup and data
import numpy as np, pandas as pd, urllib.request, time
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/main/resources/elliptic/"
for fn, remote in [("elliptic_labelled.parquet", "elliptic_labelled_v2.parquet"),
                   ("elliptic_edges_labelled.parquet", "elliptic_edges_labelled_v2.parquet")]:
    if not Path(fn).exists():
        urllib.request.urlretrieve(BASE + remote, fn)

ell = pd.read_parquet("elliptic_labelled.parquet")
edges = pd.read_parquet("elliptic_edges_labelled.parquet")
feat_cols = [c for c in ell.columns if c.startswith("feat_")]
X = ell[feat_cols].values
y = (ell["class"] == 1).astype(int).values
ts = ell["time_step"].values
boundaries = [1, 10, 20, 30, 40, 50]
windows = [(boundaries[i + 1], boundaries[i + 2]) for i in range(len(boundaries) - 2)]

assert len(y) == 46564 and y.sum() == 4545, "sample does not match the dissertation"
print(f"Transactions {len(y):,}, illicit {y.sum():,}, labelled-labelled edges {len(edges):,}")

def make_rf(seed=42):
    return Pipeline([("clf", RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight="balanced",
        random_state=seed, n_jobs=-1))])

Transactions 46,564, illicit 4,545, labelled-labelled edges 36,624


## 2. Structural facts that decide how to read the results

In [ ]:
# %% 2. Structural facts that decide how to read the results
tmap = dict(zip(ell["txId"], ell["time_step"]))
e = edges.copy()
e["t1"] = e["txId1"].map(tmap); e["t2"] = e["txId2"].map(tmap)
same_step = (e["t1"] == e["t2"]).mean()
print(f"\nShare of labelled edges joining two transactions in the SAME time step: {same_step:.4f}")
print("(If 1.0, every edge is fully observable at its own time step, so a transaction's")
print(" degree never depends on later time steps.)")

# index degrees by row position
idx = pd.Series(np.arange(len(ell)), index=ell["txId"].values)

def degrees(edge_df):
    """in/out degree per row of `ell` from an edge dataframe with txId1 -> txId2."""
    ind = edge_df["txId2"].value_counts()
    outd = edge_df["txId1"].value_counts()
    i = ell["txId"].map(ind).fillna(0).values
    o = ell["txId"].map(outd).fillna(0).values
    return i, o

def stack(i, o):
    return np.hstack([X, i.reshape(-1, 1), o.reshape(-1, 1), (i + o).reshape(-1, 1)])


Share of labelled edges joining two transactions in the SAME time step: 1.0000
(If 1.0, every edge is fully observable at its own time step, so a transaction's
 degree never depends on later time steps.)


In [ ]:
!mv txs_edgelist.csv elliptic_txs_edgelist.csv

mv: cannot stat 'txs_edgelist.csv': No such file or directory


## 3. Build the degree feature sets

In [ ]:
# %% 3. Build the degree feature sets
# A: labelled-only edges, as the dissertation did
iA, oA = degrees(edges)
XA = stack(iA, oA)

# C: notebook cell 39, rebuilt at every cutoff with edges whose BOTH endpoints are < cut.
#    Computed per window inside the loop because the matrix changes with the cutoff.
e_t = e.dropna(subset=["t1", "t2"]).copy()
e_t["edge_time"] = e_t[["t1", "t2"]].max(axis=1)

def X_C(cut):
    sub = e_t[e_t["edge_time"] < cut]
    i, o = degrees(sub)
    return stack(i, o), sub

# C2: correct time-bounded construction. A test transaction at step s may use every edge that
#     exists by step s, so edges are filtered by edge_time < hi (the END of the test window),
#     not by the training cutoff. Training rows are unaffected by this choice because their
#     edges all have edge_time < cut. If same_step == 1.0 this must equal A exactly.
def X_C2(hi):
    sub = e_t[e_t["edge_time"] < hi]
    i, o = degrees(sub)
    return stack(i, o), sub

# B: full network (all 234,355 edges incl. unlabelled neighbours); needs the Kaggle file
KAGGLE_EDGES = Path("elliptic_txs_edgelist.csv")
XB = None
if KAGGLE_EDGES.exists():
    full = pd.read_csv(KAGGLE_EDGES)
    full.columns = ["txId1", "txId2"]
    print(f"\nApproach B: full edge list loaded, {len(full):,} edges")
    assert len(full) == 234355, "unexpected edge count, wrong file?"
    iB, oB = degrees(full)      # edges are within a time step, so no timing filter is needed
    XB = stack(iB, oB)
else:
    print("\nApproach B skipped: put Kaggle's elliptic_txs_edgelist.csv in the working directory.")


Approach B: full edge list loaded, 234,355 edges


## 4. Zero-degree diagnostic: why approach C is not a valid test

In [ ]:
# %% 4. Zero-degree diagnostic: why approach C is not a valid test
print("\nShare of transactions with ZERO total degree, by construction:")
rows = []
for lo, hi in windows:
    Xc, sub = X_C(lo)
    deg = Xc[:, -1]
    tr, te = ts < lo, (ts >= lo) & (ts < hi)
    rows.append([f"t={lo}-{hi-1}", (deg[tr] == 0).mean(), (deg[te] == 0).mean(),
                 (XA[tr, -1] == 0).mean(), (XA[te, -1] == 0).mean()])
zd = pd.DataFrame(rows, columns=["window", "C train zero-deg", "C test zero-deg",
                                 "A train zero-deg", "A test zero-deg"])
print(zd.round(3).to_string(index=False))


Share of transactions with ZERO total degree, by construction:
 window  C train zero-deg  C test zero-deg  A train zero-deg  A test zero-deg
t=10-19             0.189              1.0             0.189            0.245
t=20-29             0.209              1.0             0.209            0.227
t=30-39             0.215              1.0             0.215            0.235
t=40-49             0.220              1.0             0.220            0.260


## 5. Walk-forward AUC and AP for base, A, C, C2 (and B if available), several RF seeds

In [ ]:
# %% 5. Walk-forward AUC and AP for base, A, C, C2 (and B if available), several RF seeds
SEEDS = [42, 1, 7, 21, 99]
out = []
t0 = time.time()
for seed in SEEDS:
    for lo, hi in windows:
        tr = ts < lo
        te = (ts >= lo) & (ts < hi)
        variants = {"base": X, "A": XA, "C": X_C(lo)[0], "C2": X_C2(hi)[0]}
        if XB is not None:
            variants["B"] = XB
        for name, M in variants.items():
            m = make_rf(seed).fit(M[tr], y[tr])
            p = m.predict_proba(M[te])[:, 1]
            out.append({"seed": seed, "window": f"t={lo}-{hi-1}", "variant": name,
                        "AUC": roc_auc_score(y[te], p),
                        "AP": average_precision_score(y[te], p)})
    print(f"seed {seed} done ({time.time() - t0:.0f}s)")

res = pd.DataFrame(out)
res.to_csv("e3_results_long.csv", index=False)

seed 42 done (326s)
seed 1 done (651s)
seed 7 done (982s)
seed 21 done (1314s)
seed 99 done (1644s)


## 6. Summary tables

In [ ]:
# %% 6. Summary tables
s42 = res[res.seed == 42].pivot(index="window", columns="variant", values="AUC")
cols = [c for c in ["base", "A", "B", "C", "C2"] if c in s42.columns]
print("\nWalk-forward AUC, seed 42 (as in the notebook):")
print(s42[cols].round(3).to_string())
mean42 = s42[cols].mean()
print("\nMean AUC:", mean42.round(3).to_dict())
print("Change vs base:", (mean42 - mean42["base"]).round(3).drop("base").to_dict())

print("\nAcross seeds (mean over windows, then mean and sd over seeds):")
per_seed = res.groupby(["seed", "variant"])[["AUC", "AP"]].mean().reset_index()
base_seed = per_seed[per_seed.variant == "base"].set_index("seed")
rows = []
for v in cols:
    d = per_seed[per_seed.variant == v].set_index("seed")
    dA = d["AUC"] - base_seed["AUC"]
    dP = d["AP"] - base_seed["AP"]
    rows.append([v, d["AUC"].mean(), dA.mean(), dA.std(), dP.mean(), dP.std()])
summ = pd.DataFrame(rows, columns=["variant", "mean AUC", "dAUC vs base", "sd dAUC",
                                   "dAP vs base", "sd dAP"])
print(summ.round(4).to_string(index=False))
summ.round(4).to_csv("e3_table.csv", index=False)

w = res.pivot_table(index=["window", "variant"], values="AUC", aggfunc=["mean", "std"])
print("\nPer window, mean/sd across seeds:")
print(w.round(3).to_string())


Walk-forward AUC, seed 42 (as in the notebook):
variant   base      A      B      C     C2
window                                    
t=10-19  0.995  0.995  0.995  0.995  0.995
t=20-29  0.855  0.860  0.861  0.859  0.860
t=30-39  0.990  0.990  0.990  0.990  0.990
t=40-49  0.934  0.935  0.933  0.934  0.935

Mean AUC: {'base': 0.943, 'A': 0.945, 'B': 0.945, 'C': 0.944, 'C2': 0.945}
Change vs base: {'A': 0.002, 'B': 0.002, 'C': 0.001, 'C2': 0.002}

Across seeds (mean over windows, then mean and sd over seeds):
variant  mean AUC  dAUC vs base  sd dAUC  dAP vs base  sd dAP
   base    0.9458        0.0000   0.0000       0.0000  0.0000
      A    0.9474        0.0016   0.0029       0.0003  0.0009
      B    0.9449       -0.0009   0.0074      -0.0005  0.0014
      C    0.9466        0.0008   0.0029      -0.0010  0.0009
     C2    0.9474        0.0016   0.0029       0.0003  0.0009

Per window, mean/sd across seeds:
                  mean    std
                   AUC    AUC
window  variant     

## 7. Degree distributions, illicit vs licit (the 0.40 vs 0.81 claim)

In [ ]:
# %% 7. Degree distributions, illicit vs licit (the 0.40 vs 0.81 claim)
def dist(i, o, label):
    d = pd.DataFrame({"illicit": y, "in": i, "out": o, "total": i + o})
    g = d.groupby("illicit")[["in", "out", "total"]].mean().round(2)
    g["share zero-degree"] = d.groupby("illicit")["total"].apply(lambda s: (s == 0).mean()).round(3)
    print(f"\nMean degree by class under {label}:"); print(g.to_string())
dist(iA, oA, "A (labelled-only edges)")
if XB is not None:
    dist(iB, oB, "B (full network)")


Mean degree by class under A (labelled-only edges):
           in   out  total  share zero-degree
illicit                                      
0        0.83  0.83   1.66              0.205
1        0.39  0.42   0.81              0.453

Mean degree by class under B (full network):
           in   out  total  share zero-degree
illicit                                      
0        1.91  1.19   3.10                0.0
1        1.27  0.74   2.01                0.0
